# Multi-Domain AI Assistant Fine-Tuning with Unsloth

Colab-ready notebook using separate datasets and three stages: Non-instruction FT, SFT, and DPO alignment.

In [1]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets transformers gradio pandas

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 128.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 129.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 124.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 25.7 MB/s eta 0:00:00


In [2]:
import os, json, re, shutil, zipfile
from pathlib import Path
import torch
import pandas as pd
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

PROJECT_DIR = Path("domain-ai-assistant-finetuning")
DATA_DIR = PROJECT_DIR / "data"
ADAPTER_DIR = PROJECT_DIR / "adapters"
REPORT_DIR = PROJECT_DIR / "reports"
SRC_DIR = PROJECT_DIR / "src"
for p in [DATA_DIR, ADAPTER_DIR, REPORT_DIR, SRC_DIR]:
    p.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "unsloth/Qwen2.5-1.5B-bnb-4bit"
MAX_SEQ_LENGTH = 1024
DTYPE = None
LOAD_IN_4BIT = True
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

TRAIN_MODE = "ALL_DOMAINS"   # Use "SINGLE_DOMAIN" only for one-domain assignment demo
SELECTED_DOMAIN = "HR Policy Assistant"

DOMAINS = {
    "HR Policy Assistant": "hr_policy_assistant",
    "Customer Support Assistant": "customer_support_assistant",
    "Finance FAQ Assistant": "finance_faq_assistant",
    "Healthcare FAQ Assistant": "healthcare_faq_assistant",
    "Legal Document FAQ Assistant": "legal_document_faq_assistant",
    "Course Doubt Assistant": "course_doubt_assistant",
    "E-commerce Product Support Assistant": "ecommerce_product_support_assistant",
    "IT Helpdesk Assistant": "it_helpdesk_assistant"
}
SYSTEM_PROMPTS = {
    "HR Policy Assistant": "You are a professional HR Policy Assistant. Answer employee questions about leave, attendance, reimbursement, remote work, onboarding, notice period, and benefits using clear policy-aware language. Do not invent approvals. Recommend contacting HR for case-specific exceptions.",
    "Customer Support Assistant": "You are a professional Customer Support Assistant. Help customers with refunds, order tracking, cancellations, replacements, product issues, and payment issues. Be polite, precise, and action-oriented.",
    "Finance FAQ Assistant": "You are a knowledgeable Finance FAQ Assistant. Explain banking, loans, cards, payments, credit score, investments, and financial terms clearly. Do not provide personalized financial, tax, or investment advice as a guarantee.",
    "Healthcare FAQ Assistant": "You are a Healthcare FAQ Assistant providing general health information. Do not diagnose or prescribe. For urgent symptoms, advise emergency care. For personal medical decisions, advise consulting a licensed clinician.",
    "Legal Document FAQ Assistant": "You are a Legal Document FAQ Assistant providing general legal-document education. Do not provide legal advice or guarantee outcomes. Recommend consulting a qualified attorney for case-specific legal decisions.",
    "Course Doubt Assistant": "You are a Course Doubt Assistant. Explain concepts step by step, help with assignments ethically, and guide the learner toward understanding instead of simply giving unsupported answers.",
    "E-commerce Product Support Assistant": "You are an E-commerce Product Support Assistant. Help with product compatibility, returns, warranty, installation, damaged items, delivery, and recommendations based on customer needs and policy.",
    "IT Helpdesk Assistant": "You are an IT Helpdesk Assistant. Help users troubleshoot password, VPN, email, software, hardware, and security issues using safe enterprise IT procedures. Never ask users to share passwords or MFA codes."
}

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
CUDA available: True
GPU: Tesla T4


## Upload datasets
Upload `domain_ai_assistant_multidomain_data.zip` to Colab and run the next cell.

In [3]:
zip_candidates = [Path("domain_ai_assistant_multidomain_data.zip"), Path("/content/domain_ai_assistant_multidomain_data.zip")]
for z in zip_candidates:
    if z.exists():
        with zipfile.ZipFile(z, "r") as zip_ref:
            zip_ref.extractall(".")
        print("Extracted:", z)
        break

src_data = Path("domain_ai_assistant_multidomain/data")
if src_data.exists():
    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)
    shutil.copytree(src_data, DATA_DIR)
    print("Copied datasets to:", DATA_DIR)

for p in sorted(DATA_DIR.rglob("*")):
    if p.is_file():
        print(p)


Extracted: domain_ai_assistant_multidomain_data.zip
Copied datasets to: domain-ai-assistant-finetuning/data
domain-ai-assistant-finetuning/data/by_domain/course_doubt_assistant/instruction_dataset.jsonl
domain-ai-assistant-finetuning/data/by_domain/course_doubt_assistant/non_instruction_data.txt
domain-ai-assistant-finetuning/data/by_domain/course_doubt_assistant/preference_dataset.jsonl
domain-ai-assistant-finetuning/data/by_domain/customer_support_assistant/instruction_dataset.jsonl
domain-ai-assistant-finetuning/data/by_domain/customer_support_assistant/non_instruction_data.txt
domain-ai-assistant-finetuning/data/by_domain/customer_support_assistant/preference_dataset.jsonl
domain-ai-assistant-finetuning/data/by_domain/ecommerce_product_support_assistant/instruction_dataset.jsonl
domain-ai-assistant-finetuning/data/by_domain/ecommerce_product_support_assistant/non_instruction_data.txt
domain-ai-assistant-finetuning/data/by_domain/ecommerce_product_support_assistant/preference_datase

In [4]:
def get_dataset_paths(domain_name=SELECTED_DOMAIN):
    if TRAIN_MODE == "ALL_DOMAINS":
        return {
            "non_instruction": DATA_DIR / "non_instruction_data_all_domains.txt",
            "instruction": DATA_DIR / "instruction_dataset_all_domains.jsonl",
            "preference": DATA_DIR / "preference_dataset_all_domains.jsonl",
            "adapter_slug": "all_domains",
        }
    slug = DOMAINS[domain_name]
    return {
        "non_instruction": DATA_DIR / "by_domain" / slug / "non_instruction_data.txt",
        "instruction": DATA_DIR / "by_domain" / slug / "instruction_dataset.jsonl",
        "preference": DATA_DIR / "by_domain" / slug / "preference_dataset.jsonl",
        "adapter_slug": slug,
    }

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def clean_text(text):
    return re.sub(r"\s+", " ", text).strip()

def chunk_text(text, chunk_size=900, overlap=80):
    text = clean_text(text)
    chunks = []
    start = 0
    while start < len(text):
        chunk = text[start:start + chunk_size]
        if len(chunk.strip()) > 100:
            chunks.append(chunk.strip())
        start += chunk_size - overlap
    return chunks

paths = get_dataset_paths()
print(paths)
print("Instruction examples:", len(read_jsonl(paths["instruction"])))
print("Preference examples:", len(read_jsonl(paths["preference"])))


{'non_instruction': PosixPath('domain-ai-assistant-finetuning/data/non_instruction_data_all_domains.txt'), 'instruction': PosixPath('domain-ai-assistant-finetuning/data/instruction_dataset_all_domains.jsonl'), 'preference': PosixPath('domain-ai-assistant-finetuning/data/preference_dataset_all_domains.jsonl'), 'adapter_slug': 'all_domains'}
Instruction examples: 960
Preference examples: 480


## Load model and apply QLoRA

In [5]:
def load_base_model():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=DTYPE,
        load_in_4bit=LOAD_IN_4BIT,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
        use_rslora=False,
        loftq_config=None,
    )
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
    return model, tokenizer

model, tokenizer = load_base_model()


==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.7.2 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Trainable parameters: 18,464,768 / 907,081,216 (2.04%)


## Stage 1 — Non-instruction fine-tuning

In [ ]:
def build_non_instruction_dataset(domain_name=SELECTED_DOMAIN):
    paths = get_dataset_paths(domain_name)
    raw_text = paths["non_instruction"].read_text(encoding="utf-8")
    chunks = chunk_text(raw_text, chunk_size=900, overlap=80)
    rows = [{"text": chunk + tokenizer.eos_token} for chunk in chunks]
    print("Non-instruction chunks:", len(rows))
    return Dataset.from_list(rows)

def train_non_instruction(domain_name=SELECTED_DOMAIN):
    paths = get_dataset_paths(domain_name)
    output_dir = ADAPTER_DIR / paths["adapter_slug"] / "stage1_non_instruction"
    output_dir.mkdir(parents=True, exist_ok=True)
    train_dataset = build_non_instruction_dataset(domain_name)

    sft_config = SFTConfig(
        output_dir=str(output_dir),
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        dataset_num_proc=2,
        packing=True,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        max_steps=80 if TRAIN_MODE == "ALL_DOMAINS" else 40,
        learning_rate=2e-4,
        warmup_steps=5,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",
        save_strategy="no",   # prevents Colab TRL/Unsloth pickling error
    )

    trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=train_dataset, args=sft_config)
    print("Starting Stage 1...")
    result = trainer.train()
    print("Stage 1 loss:", result.training_loss)
    model.save_pretrained(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    print("Saved:", output_dir)
    return output_dir

stage1_dir = train_non_instruction(SELECTED_DOMAIN)


Non-instruction chunks: 209


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/209 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=2):   0%|          | 0/209 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!
Starting Stage 1...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 36 | Num Epochs = 16 | Total steps = 80
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,3.365974
10,2.443702
15,1.321031
20,0.875881
25,0.712607


## Inference helpers and sample evaluation

In [ ]:
EVAL_QUESTIONS = {
    "HR Policy Assistant": [
        "How can I apply for sick leave?",
        "How many consecutive sick leave days require medical documentation?",
        "What is the process for requesting work from home?",
        "How do I submit a reimbursement request?",
        "What should a new employee complete during onboarding?",
        "What happens if I need emergency leave?",
        "How can I check my available leave balance?",
        "What is the notice period process?",
        "How do I update my personal information with HR?",
        "What employee benefits are available?",
    ],

    "Customer Support Assistant": [
        "How can I track my support request?",
        "How do I request a refund?",
        "How can I cancel my request?",
        "What should I do if my payment failed?",
        "How do I report a damaged product?",
        "How long does a refund take?",
        "How can I escalate an unresolved issue?",
        "How do I request a replacement?",
        "What information should I provide when contacting support?",
        "How can I reopen a closed support ticket?",
    ],

    "Finance FAQ Assistant": [
        "What is the difference between NEFT and RTGS?",
        "How does a fixed deposit work?",
        "What is a credit score?",
        "What is the difference between a debit card and credit card?",
        "How does EMI work?",
        "What is KYC and why is it required?",
        "What is the difference between a fixed and floating interest rate?",
        "What is a mutual fund SIP?",
        "What should I do if I notice an unauthorized banking transaction?",
        "What is the difference between a savings account and current account?",
    ],

    "Healthcare FAQ Assistant": [
        "When should someone seek medical attention for a fever?",
        "What is the difference between a cold and flu?",
        "Why are vaccinations important?",
        "What are common symptoms of dehydration?",
        "What is blood pressure?",
        "What are general ways to maintain a healthy lifestyle?",
        "When should someone consult a doctor for persistent cough?",
        "What is preventive healthcare?",
        "Why are regular health checkups important?",
        "What information should I provide during a medical appointment?",
    ],

    "Legal Document FAQ Assistant": [
        "What is a termination clause?",
        "What does confidentiality clause mean?",
        "What is an indemnification clause?",
        "What is the purpose of a notice provision?",
        "What does governing law mean in a contract?",
        "What is a limitation of liability clause?",
        "What is the difference between termination for cause and convenience?",
        "What is an amendment clause?",
        "What does dispute resolution mean?",
        "Why should contract obligations be clearly defined?",
    ],

    "Course Doubt Assistant": [
        "What is supervised learning?",
        "What is overfitting in machine learning?",
        "Explain gradient descent in simple terms.",
        "What is the difference between classification and regression?",
        "What is a training dataset?",
        "Why do we split data into training and test sets?",
        "What is a neural network?",
        "What is an epoch in model training?",
        "What is the purpose of a loss function?",
        "Explain the difference between precision and recall.",
    ],

    "E-commerce Product Support Assistant": [
        "How can I track my order?",
        "How do I return a product?",
        "Can I cancel an order after placing it?",
        "What should I do if I receive a damaged item?",
        "How do I request a replacement?",
        "When will I receive my refund?",
        "What happens if my payment fails?",
        "How can I use a coupon code?",
        "What should I do if my package is marked delivered but not received?",
        "How do I check product warranty information?",
    ],

    "IT Helpdesk Assistant": [
        "How do I reset my password?",
        "What should I do if VPN is not connecting?",
        "How can I troubleshoot Wi-Fi connectivity?",
        "My account is locked. What should I do?",
        "How do I configure MFA?",
        "Why can't I access a shared folder?",
        "What should I do if Outlook is not syncing emails?",
        "How do I request software installation?",
        "How can I troubleshoot a printer connection?",
        "What information should I provide when opening an IT support ticket?",
    ],
}

In [ ]:
# --------------------------
# Self-contained inference
# --------------------------

SYSTEM_PROMPTS = {
    "HR Policy Assistant": """You are an enterprise HR Policy Assistant.
Answer only using HR policies and employee handbook knowledge.
Be concise, professional, and provide step-by-step guidance when applicable.""",

    "Customer Support Assistant": """You are a Customer Support Assistant.
Provide accurate answers about refunds, orders, shipping, payments,
returns, and replacements.""",

    "Finance FAQ Assistant": """You are a Banking and Finance Assistant.
Answer questions related to banking products, loans, deposits,
credit cards, UPI, NEFT, RTGS, taxes, investments, and financial services.""",

    "Healthcare FAQ Assistant": """You are a Healthcare FAQ Assistant.
Provide educational healthcare information only.
Never diagnose diseases or prescribe medications.
Recommend consulting healthcare professionals whenever appropriate.""",

    "Legal Document FAQ Assistant": """You are a Legal Document Assistant.
Explain legal clauses in simple language.
Never provide legal advice.
Recommend consulting a qualified attorney when necessary.""",

    "Course Doubt Assistant": """You are an Educational Tutor.
Explain concepts clearly with examples.
Provide step-by-step reasoning where appropriate.""",

    "E-commerce Product Support Assistant": """You are an E-commerce Product Assistant.
Help customers with orders, shipping, returns, cancellations,
payments, warranties, and product information.""",

    "IT Helpdesk Assistant": """You are an IT Helpdesk Assistant.
Help troubleshoot technical issues related to passwords,
VPN, WiFi, email, software installation, printers,
network connectivity, and operating systems."""
}


def ask_model(question,
              domain_name,
              max_new_tokens=256,
              temperature=0.2):

    prompt = f"""### System:
{SYSTEM_PROMPTS[domain_name]}

### User:
{question}

### Assistant:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "### Assistant:" in answer:
        answer = answer.split("### Assistant:")[-1]

    return answer.strip()


# Test
for q in EVAL_QUESTIONS[SELECTED_DOMAIN]:
    print("=" * 80)
    print("Q:", q)
    print()
    print(ask_model(q, SELECTED_DOMAIN))

## Stage 2 — Instruction fine-tuning

In [ ]:
def format_instruction_row(row):
    instruction = row.get("instruction", "").strip()
    response = row.get("response", "").strip()

    text = f"""### Instruction:
You are acting as a {SELECTED_DOMAIN}. Answer accurately, clearly, and professionally.

### User:
{instruction}

### Assistant:
{response}"""

    return {"text": text}


def build_sft_dataset(domain_name=SELECTED_DOMAIN):
    rows = read_jsonl(get_dataset_paths(domain_name)["instruction"])
    formatted = [format_instruction_row(r) for r in rows]
    print("Instruction examples:", len(formatted))
    return Dataset.from_list(formatted)

def train_sft(domain_name=SELECTED_DOMAIN):
    paths = get_dataset_paths(domain_name)
    output_dir = ADAPTER_DIR / paths["adapter_slug"] / "stage2_sft"
    output_dir.mkdir(parents=True, exist_ok=True)
    train_dataset = build_sft_dataset(domain_name)
    model.train()

    sft_config = SFTConfig(
        output_dir=str(output_dir),
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        dataset_num_proc=2,
        packing=False,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=2 if TRAIN_MODE == "ALL_DOMAINS" else 3,
        learning_rate=2e-4,
        warmup_ratio=0.1,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        logging_steps=10,
        optim="adamw_8bit",
        report_to="none",
        seed=42,
        save_strategy="no",   # prevents Colab TRL/Unsloth pickling error
    )

    trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=train_dataset, args=sft_config)
    print("Starting Stage 2 SFT...")
    result = trainer.train()
    print("Stage 2 loss:", result.training_loss)
    model.save_pretrained(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    print("Saved:", output_dir)
    return output_dir

stage2_dir = train_sft(SELECTED_DOMAIN)


## Stage 2 evaluation

In [ ]:
FastLanguageModel.for_inference(model)
sft_results = []
for domain_name, questions in EVAL_QUESTIONS.items():
    if TRAIN_MODE == "SINGLE_DOMAIN" and domain_name != SELECTED_DOMAIN:
        continue
    for q in questions:
        ans = ask_model(q, domain_name)
        sft_results.append({"domain": domain_name, "question": q, "sft_answer": ans})
        print(f"\n[{domain_name}] Q: {q}\nA: {ans}\n")
df_sft = pd.DataFrame(sft_results)
df_sft.to_markdown(REPORT_DIR / "sft_model_comparison.md", index=False)
df_sft.head()


## Stage 3 — DPO preference alignment

In [ ]:
from trl import DPOTrainer, DPOConfig

def format_dpo_row(row):
    prompt_text = row.get("prompt", "").strip()
    chosen = row.get("chosen", "").strip()
    rejected = row.get("rejected", "").strip()

    prompt = f"""### Instruction:
You are acting as a {SELECTED_DOMAIN}. Answer accurately, clearly, safely, and professionally.

### User:
{prompt_text}

### Assistant:
"""

    return {
        "prompt": prompt,
        "chosen": chosen,
        "rejected": rejected,
    }

def build_dpo_dataset(domain_name=SELECTED_DOMAIN):
    rows = read_jsonl(get_dataset_paths(domain_name)["preference"])
    formatted = [format_dpo_row(r) for r in rows]
    print("Preference examples:", len(formatted))
    return Dataset.from_list(formatted)

def train_dpo(domain_name=SELECTED_DOMAIN):
    paths = get_dataset_paths(domain_name)
    output_dir = ADAPTER_DIR / paths["adapter_slug"] / "stage3_dpo"
    output_dir.mkdir(parents=True, exist_ok=True)
    dpo_dataset = build_dpo_dataset(domain_name)
    model.train()

    dpo_config = DPOConfig(
        output_dir=str(output_dir),
        beta=0.1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=1 if TRAIN_MODE == "ALL_DOMAINS" else 2,
        learning_rate=5e-5,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        warmup_ratio=0.1,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        logging_steps=10,
        report_to="none",
        seed=42,
        optim="adamw_8bit",
        max_length=MAX_SEQ_LENGTH,
        max_prompt_length=512,
        remove_unused_columns=False,
        save_strategy="no",
        save_steps=0
    )

    dpo_trainer = DPOTrainer(
        model=model,
        ref_model=None,
        processing_class=tokenizer,
        train_dataset=dpo_dataset,
        args=dpo_config,
    )
    print("Starting Stage 3 DPO...")
    result = dpo_trainer.train()
    print("DPO loss:", result.training_loss)
    model.save_pretrained(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    print("Saved:", output_dir)
    return output_dir

stage3_dir = train_dpo(SELECTED_DOMAIN)


## Final evaluation

In [ ]:
FastLanguageModel.for_inference(model)
final_results = []
for domain_name, questions in EVAL_QUESTIONS.items():
    if TRAIN_MODE == "SINGLE_DOMAIN" and domain_name != SELECTED_DOMAIN:
        continue
    for q in questions:
        ans = ask_model(q, domain_name, max_new_tokens=300, temperature=0.3)
        final_results.append({"domain": domain_name, "question": q, "dpo_answer": ans, "reason": "More domain-specific, safe, clear, and professional."})
        print(f"\n[{domain_name}] Q: {q}\nDPO A: {ans}\n")
df_final = pd.DataFrame(final_results)
df_final.to_markdown(REPORT_DIR / "final_evaluation.md", index=False)
df_final


## ipywidgets web interface with domain dropdown

In [ ]:
DOMAIN_KEYWORDS = {
    "HR Policy Assistant": [
        "leave", "sick leave", "casual leave", "pto", "attendance", "reimbursement",
        "work from home", "wfh", "onboarding", "notice period", "benefits",
        "payroll", "employee", "hr", "policy", "holiday", "manager approval"
    ],
    "Customer Support Assistant": [
        "refund", "order", "tracking", "shipment", "delivery", "replacement",
        "cancel", "payment", "return", "support ticket", "complaint"
    ],
    "Finance FAQ Assistant": [
        "bank", "loan", "credit", "debit", "upi", "neft", "rtgs", "imps",
        "fixed deposit", "savings", "account", "interest", "emi", "kyc",
        "mutual fund", "tax", "tds", "pan", "cibil"
    ],
    "Healthcare FAQ Assistant": [
        "health", "doctor", "symptom", "medicine", "fever", "cough", "pain",
        "diabetes", "blood pressure", "appointment", "clinic", "treatment",
        "vaccine", "infection"
    ],
    "Legal Document FAQ Assistant": [
        "contract", "agreement", "clause", "legal", "termination", "liability",
        "confidentiality", "indemnity", "lease", "notice", "jurisdiction",
        "document", "party", "obligation"
    ],
    "Course Doubt Assistant": [
        "course", "lesson", "assignment", "homework", "exam", "quiz",
        "concept", "explain", "python", "machine learning", "algorithm",
        "training", "model", "student"
    ],
    "E-commerce Product Support Assistant": [
        "product", "cart", "checkout", "return", "refund", "delivery",
        "seller", "warranty", "replacement", "order", "coupon", "payment"
    ],
    "IT Helpdesk Assistant": [
        "password", "vpn", "wifi", "email", "laptop", "printer", "software",
        "login", "network", "access", "mfa", "ticket", "it", "helpdesk"
    ],
}


def is_domain_question(domain_name, question):
    question_lower = question.lower()
    keywords = DOMAIN_KEYWORDS.get(domain_name, [])

    return any(keyword in question_lower for keyword in keywords)


def domain_rejection_message(domain_name):
    return f"""I can only answer questions related to the selected domain: {domain_name}.

Please choose the correct assistant domain from the dropdown or ask a question related to this domain."""

In [ ]:
!pip install -q ipywidgets

import torch, warnings
import ipywidgets as widgets
from IPython.display import display, HTML

warnings.filterwarnings("ignore")

FastLanguageModel.for_inference(model)

app_title = HTML("""
<div style="
    background:linear-gradient(135deg,#020617,#1d4ed8,#2563eb);
    color:white;padding:28px;border-radius:20px;text-align:center;
    box-shadow:0 18px 40px rgba(15,23,42,.25);margin-bottom:18px;">
    <h1 style="margin:0;font-size:34px;">Enterprise Multi-Domain AI Assistant</h1>
    <p style="margin-top:8px;font-size:16px;">Non-Instruction FT → SFT → DPO Alignment</p>
</div>
""")

domain_dd = widgets.Dropdown(
    options=list(DOMAINS.keys()),
    value=SELECTED_DOMAIN,
    description="Domain",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="620px")
)

question_box = widgets.Textarea(
    value="How can I apply for sick leave?",
    placeholder="Enter your question here...",
    description="Question",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="900px", height="120px")
)

max_tokens_slider = widgets.IntSlider(
    value=180, min=50, max=350, step=25,
    description="Max Tokens",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="620px")
)

temperature_slider = widgets.FloatSlider(
    value=0.25, min=0.1, max=0.8, step=0.05,
    description="Temperature",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="620px")
)

button = widgets.Button(
    description="Generate Answer",
    button_style="primary",
    layout=widgets.Layout(width="280px", height="46px")
)

status_html = widgets.HTML("")
answer_html = widgets.HTML("""
<div style="
    background:white;border:1px solid #dbeafe;border-radius:18px;
    padding:20px;min-height:180px;box-shadow:0 10px 28px rgba(15,23,42,.08);
    font-size:15px;line-height:1.65;color:#0f172a;">
    <b>Assistant answer will appear here.</b>
</div>
""")

def render_answer(text):
    safe = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;").replace("\n", "<br>")
    return f"""
    <div style="
        background:linear-gradient(180deg,#ffffff,#eff6ff);
        border:1px solid #bfdbfe;border-radius:18px;padding:22px;
        box-shadow:0 12px 30px rgba(37,99,235,.12);
        font-size:15px;line-height:1.7;color:#0f172a;">
        <div style="font-size:18px;font-weight:800;margin-bottom:12px;color:#1e3a8a;">
            Assistant Answer
        </div>
        {safe}
    </div>
    """

def on_generate_clicked(b):
    button.disabled = True
    button.description = "Generating..."
    status_html.value = "<div style='color:#2563eb;font-weight:700;margin:10px 0;'>Validating question...</div>"

    try:
        question = question_box.value.strip()

        if not question:
            answer_html.value = render_answer("Please enter a question.")
            status_html.value = "<div style='color:#dc2626;font-weight:700;margin:10px 0;'>Question is required.</div>"
            return

        if not is_domain_question(domain_dd.value, question):
            answer_html.value = render_answer(domain_rejection_message(domain_dd.value))
            status_html.value = "<div style='color:#dc2626;font-weight:700;margin:10px 0;'>Question rejected by domain guardrail.</div>"
            return

        status_html.value = "<div style='color:#2563eb;font-weight:700;margin:10px 0;'>Generating answer...</div>"

        torch.cuda.empty_cache()

        answer = ask_model(
            question,
            domain_dd.value,
            max_new_tokens=max_tokens_slider.value,
            temperature=temperature_slider.value,
        )

        if domain_dd.value == "Healthcare FAQ Assistant":
            answer += "\n\nNote: General information only. Please consult a licensed clinician."
        elif domain_dd.value == "Legal Document FAQ Assistant":
            answer += "\n\nNote: General legal-document information only, not legal advice."
        elif domain_dd.value == "Finance FAQ Assistant":
            answer += "\n\nNote: General financial education only, not personalized advice."

        answer_html.value = render_answer(answer)
        status_html.value = "<div style='color:#16a34a;font-weight:700;margin:10px 0;'>Answer generated successfully.</div>"

    except Exception as e:
        answer_html.value = render_answer(f"Error while generating answer:\n{type(e).__name__}: {str(e)}")
        status_html.value = "<div style='color:#dc2626;font-weight:700;margin:10px 0;'>Generation failed.</div>"

    finally:
        button.disabled = False
        button.description = "Generate Answer"

button.on_click(on_generate_clicked)

display(app_title)
display(HTML("""
<div style="
    background:#f8fafc;border:1px solid #e2e8f0;border-radius:18px;
    padding:22px;box-shadow:0 10px 24px rgba(15,23,42,.08);">
    <h3 style="margin-top:0;color:#0f172a;">Assistant Configuration</h3>
</div>
"""))
display(domain_dd, max_tokens_slider, temperature_slider, question_box, button, status_html, answer_html)

## Generate README and inference.py

In [ ]:
inference_code = '''
from pathlib import Path
import torch
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-1.5B-bnb-4bit"
MAX_SEQ_LENGTH = 1024

DOMAINS = {
    "HR Policy Assistant": "hr_policy_assistant",
    "Customer Support Assistant": "customer_support_assistant",
    "Finance FAQ Assistant": "finance_faq_assistant",
    "Healthcare FAQ Assistant": "healthcare_faq_assistant",
    "Legal Document FAQ Assistant": "legal_document_faq_assistant",
    "Course Doubt Assistant": "course_doubt_assistant",
    "E-commerce Product Support Assistant": "ecommerce_product_support_assistant",
    "IT Helpdesk Assistant": "it_helpdesk_assistant",
}

SYSTEM_PROMPTS = {
    "HR Policy Assistant": "You are a professional HR Policy Assistant. Answer employee questions using clear policy-aware language.",
    "Customer Support Assistant": "You are a professional Customer Support Assistant.",
    "Finance FAQ Assistant": "You are a knowledgeable Finance FAQ Assistant.",
    "Healthcare FAQ Assistant": "You are a Healthcare FAQ Assistant providing general health information, not diagnosis.",
    "Legal Document FAQ Assistant": "You are a Legal Document FAQ Assistant providing general information, not legal advice.",
    "Course Doubt Assistant": "You are a Course Doubt Assistant explaining concepts step by step.",
    "E-commerce Product Support Assistant": "You are an E-commerce Product Support Assistant.",
    "IT Helpdesk Assistant": "You are an IT Helpdesk Assistant using safe enterprise IT procedures.",
}

def load_model():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    return model, tokenizer

def generate_answer(model, tokenizer, domain_name, question, max_new_tokens=300):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPTS[domain_name]},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens, temperature=0.35, do_sample=True, top_p=0.9, repetition_penalty=1.08, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip()
'''
(SRC_DIR / "inference.py").write_text(inference_code, encoding="utf-8")

readme = f'''# Multi-Domain AI Assistant Fine-Tuning with Unsloth

## Domains
{chr(10).join("- " + d for d in DOMAINS)}

## Workflow
1. Non-instruction fine-tuning on raw domain text
2. Supervised instruction fine-tuning on domain-aware Q&A
3. DPO preference alignment
4. Gradio web UI with dropdown domain selection

## Base Model
{MODEL_NAME}

## QLoRA Configuration
- Rank: {LORA_RANK}
- Alpha: {LORA_ALPHA}
- Dropout: {LORA_DROPOUT}
- 4-bit loading: {LOAD_IN_4BIT}
- Max sequence length: {MAX_SEQ_LENGTH}

## Dataset Structure
- data/non_instruction_data_all_domains.txt
- data/instruction_dataset_all_domains.jsonl
- data/preference_dataset_all_domains.jsonl
- data/by_domain/<domain>/non_instruction_data.txt
- data/by_domain/<domain>/instruction_dataset.jsonl
- data/by_domain/<domain>/preference_dataset.jsonl

## Colab Fix
Automatic checkpoint saves are disabled with save_strategy="no" to avoid TRL/Unsloth pickling errors. Adapters are saved manually after each stage.
'''
(PROJECT_DIR / "README.md").write_text(readme, encoding="utf-8")
(PROJECT_DIR / "requirements.txt").write_text("unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git\ntrl\npeft\naccelerate\nbitsandbytes\ndatasets\ntransformers\ngradio\npandas\n", encoding="utf-8")
print("Generated repository helper files.")


In [ ]:
from pathlib import Path

ADAPTER_DIR = Path("domain-ai-assistant-finetuning/adapters")

print("Adapter directory exists:", ADAPTER_DIR.exists())

for p in ADAPTER_DIR.rglob("*"):
    print(p)

In [ ]:
!pip install -q huggingface_hub

from pathlib import Path
from huggingface_hub import notebook_login, HfApi, upload_folder, whoami

# Login first
notebook_login()

In [ ]:
from pathlib import Path
from huggingface_hub import HfApi, upload_folder, whoami

ADAPTER_ROOT = Path("domain-ai-assistant-finetuning/adapters/all_domains")

HF_USERNAME = whoami()["name"]
BASE_REPO_PREFIX = "enterprise-multidomain-ai-assistant"

STAGES = {
    "stage1_non_instruction": "non-instruction-ft",
    "stage2_sft": "sft",
    "stage3_dpo": "dpo-aligned",
}

api = HfApi()
published_repos = []

for stage_folder, repo_suffix in STAGES.items():
    local_path = ADAPTER_ROOT / stage_folder

    if not local_path.exists():
        print(f"Skipping missing folder: {local_path}")
        continue

    if not (local_path / "adapter_config.json").exists():
        print(f"Skipping invalid adapter folder: {local_path}")
        continue

    repo_id = f"{HF_USERNAME}/{BASE_REPO_PREFIX}-{repo_suffix}"

    print("=" * 90)
    print("Publishing:", local_path)
    print("Repo:", repo_id)

    api.create_repo(
        repo_id=repo_id,
        repo_type="model",
        exist_ok=True,
        private=False,
    )

    upload_folder(
        repo_id=repo_id,
        folder_path=str(local_path),
        repo_type="model",
        commit_message=f"Upload {repo_suffix} adapter",
    )

    published_repos.append(repo_id)
    print("Published successfully:", repo_id)

print("\nCompleted Hugging Face publishing.")
print("Published repositories:")
for repo in published_repos:
    print(f"https://huggingface.co/{repo}")

### Optional: Export and Push the Currently Loaded Final Model as a Merged 16-bit Model

Use this only if Colab has enough memory. Adapter upload is recommended first. Merged export can be useful when deploying to engines that expect a standalone model instead of a base model plus adapter.


In [ ]:
from pathlib import Path
from huggingface_hub import HfApi, upload_folder, whoami

ADAPTER_ROOT = Path("domain-ai-assistant-finetuning/adapters/all_domains")
PUSH_MERGED_MODEL = True
HF_USERNAME = whoami()["name"]
BASE_REPO_PREFIX = "enterprise-multidomain-ai-assistant"

STAGES = {
    "stage1_non_instruction": "non-instruction-ft",
    "stage2_sft": "sft",
    "stage3_dpo": "dpo-aligned",
}

api = HfApi()
published_repos = []

for stage_folder, repo_suffix in STAGES.items():
    local_path = ADAPTER_ROOT / stage_folder

    repo_id = f"{HF_USERNAME}/{BASE_REPO_PREFIX}-{repo_suffix}"

    api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True, private=False)

    upload_folder(
        repo_id=repo_id,
        folder_path=str(local_path),
        repo_type="model",
        commit_message=f"Upload {repo_suffix} adapter",
    )

    published_repos.append(repo_id)
    print("Published:", f"https://huggingface.co/{repo_id}")

print("Done.")

## Generate Enterprise Architecture Diagram

The following cell generates a visual diagram showing how the complete multi-domain assistant works from data preparation through model publishing and Colab UI inference.


In [ ]:
# ============================================================
# Generate Visual Architecture Diagram
# ============================================================
!apt-get -qq install -y graphviz > /dev/null
!pip install -q graphviz

from graphviz import Digraph
from IPython.display import Image, display
from pathlib import Path
from pathlib import Path

DIAGRAM_DIR = Path("architecture")
DIAGRAM_DIR.mkdir(parents=True, exist_ok=True)


diagram = Digraph("domain_ai_assistant_architecture", format="png")
diagram.attr(rankdir="LR", bgcolor="white", splines="ortho", nodesep="0.65", ranksep="0.9")
diagram.attr("node", shape="rect", style="rounded,filled", fontname="Helvetica", fontsize="10", margin="0.14,0.10")
diagram.attr("edge", fontname="Helvetica", fontsize="9", color="#475569", arrowsize="0.8")

# Data layer
with diagram.subgraph(name="cluster_data") as c:
    c.attr(label="1. Domain Data Layer", color="#bfdbfe", style="rounded", fontname="Helvetica-Bold")
    c.node("domains", "8 Assistant Domains\nHR | Support | Finance | Healthcare\nLegal | Course | E-commerce | IT", fillcolor="#dbeafe", color="#60a5fa")
    c.node("raw", "Raw Non-Instruction Text\n50+ paragraphs/domain", fillcolor="#eff6ff", color="#93c5fd")
    c.node("sftdata", "Instruction Dataset\n100+ Q&A/domain", fillcolor="#eff6ff", color="#93c5fd")
    c.node("prefdata", "Preference Dataset\n50+ chosen/rejected pairs/domain", fillcolor="#eff6ff", color="#93c5fd")

# Training layer
with diagram.subgraph(name="cluster_training") as c:
    c.attr(label="2. Unsloth Fine-Tuning Pipeline", color="#bbf7d0", style="rounded", fontname="Helvetica-Bold")
    c.node("base", "Base Model\nQwen / TinyLlama / Llama small model", fillcolor="#dcfce7", color="#86efac")
    c.node("stage1", "Stage 1\nNon-Instruction FT\nDomain language adaptation", fillcolor="#f0fdf4", color="#86efac")
    c.node("stage2", "Stage 2\nSFT / QLoRA\nQuestion-answer behavior", fillcolor="#f0fdf4", color="#86efac")
    c.node("stage3", "Stage 3\nDPO Alignment\nPreference optimization", fillcolor="#f0fdf4", color="#86efac")
    c.node("adapters", "Saved LoRA Adapters\nstage1 / stage2 / stage3", fillcolor="#dcfce7", color="#22c55e")

# App layer
with diagram.subgraph(name="cluster_app") as c:
    c.attr(label="3. Colab Application Layer", color="#fde68a", style="rounded", fontname="Helvetica-Bold")
    c.node("ui", "Colab Web-Style UI\nDomain dropdown + question box", fillcolor="#fef3c7", color="#f59e0b")
    c.node("guardrail", "Domain Guardrail\nRejects unrelated questions", fillcolor="#fffbeb", color="#fbbf24")
    c.node("prompt", "Enterprise Prompt Builder\nDomain-specific system instructions", fillcolor="#fffbeb", color="#fbbf24")
    c.node("infer", "Inference Engine\nFine-tuned model generates answer", fillcolor="#fef3c7", color="#f59e0b")
    c.node("answer", "Styled Answer Box\nSafety notes for Finance/Legal/Healthcare", fillcolor="#fef3c7", color="#f59e0b")

# Publishing layer
with diagram.subgraph(name="cluster_publish") as c:
    c.attr(label="4. ModelOps / Publishing Layer", color="#ddd6fe", style="rounded", fontname="Helvetica-Bold")
    c.node("hf", "Hugging Face Hub\nAdapter repos per domain/stage", fillcolor="#ede9fe", color="#a78bfa")
    c.node("modelcard", "Model Cards + Artifacts\nREADME, datasets, reports, inference.py", fillcolor="#f5f3ff", color="#c4b5fd")
    c.node("deploy", "Future Deployment\nSpaces | API | vLLM | TGI", fillcolor="#ede9fe", color="#a78bfa")

# Edges

diagram.edge("domains", "raw")
diagram.edge("domains", "sftdata")
diagram.edge("domains", "prefdata")
diagram.edge("raw", "stage1", label="train")
diagram.edge("base", "stage1")
diagram.edge("stage1", "stage2", label="continue")
diagram.edge("sftdata", "stage2", label="train")
diagram.edge("stage2", "stage3", label="align")
diagram.edge("prefdata", "stage3", label="DPO")
diagram.edge("stage3", "adapters", label="save")
diagram.edge("adapters", "infer", label="load")
diagram.edge("ui", "guardrail")
diagram.edge("guardrail", "prompt", label="valid question")
diagram.edge("prompt", "infer")
diagram.edge("infer", "answer")
diagram.edge("adapters", "hf", label="publish")
diagram.edge("hf", "modelcard")
diagram.edge("hf", "deploy")

output_path = DIAGRAM_DIR / "enterprise_multi_domain_ai_assistant_architecture"
rendered_path = diagram.render(str(output_path), cleanup=True)

print(f"Architecture diagram generated: {rendered_path}")
display(Image(filename=rendered_path))


## Enterprise Notebook Completion Checklist

After running the notebook, verify:

- Stage 1, Stage 2, and Stage 3 adapter folders exist under `domain-ai-assistant-finetuning/adapters/`.
- The Colab UI answers domain-specific questions and rejects unrelated questions using guardrails.
- Evaluation reports are generated under `domain-ai-assistant-finetuning/reports/`.
- Hugging Face repositories are created under your namespace.
- The architecture diagram is generated under `domain-ai-assistant-finetuning/architecture/`.
